# Lesson 19: Models-as-Tools — Task-Based SLM Routers, Hybrid Gemini Fallback & Multi-Agent Edge Coordinators

In this companion notebook, you will:
1. Build a **Models-as-Tools** registry mapping `general`, `reasoning`, `code`, and `creative` intents to specialized local SLMs.
2. Implement **Hybrid Complexity Escalation** that routes privacy-sensitive or short tasks locally and escalates complex synthesis to **Google AI Studio (`gemini-2.5-flash`)** when `GEMINI_API_KEY` is configured.
3. Run a **Coordinator + 3-Specialist** (Retrieval, Reasoning, Execution) multi-agent pipeline with refinement feedback.

In [ ]:
!pip install -q -U google-genai

## 1. Intelligent Task Router & Hybrid Escalation
Route queries deterministically by domain keywords and complexity threshold.

In [ ]:
import os

TOOL_REGISTRY = {
    "general":   {"model": "phi-4-mini",   "target": "edge",  "temp": 0.5},
    "code":      {"model": "qwen2.5-7b",   "target": "edge",  "temp": 0.2},
    "reasoning": {"model": "phi-4-mini",   "target": "edge",  "temp": 0.3},
    "research":  {"model": "gemini-2.5-flash", "target": "cloud", "temp": 0.4},
}

def route_query(prompt: str, privacy_strict: bool = False) -> dict:
    p = prompt.lower()
    if any(k in p for k in ("python", "bug", "refactor", "sql")):
        route = "code"
    elif any(k in p for k in ("compare", "architecture", "multi-year")) and not privacy_strict:
        route = "research"
    elif any(k in p for k in ("derive", "prove", "step")):
        route = "reasoning"
    else:
        route = "general"
    return {"route": route, **TOOL_REGISTRY[route]}

for q in [
    "Refactor this Python loop for ONNX export",
    "Compare global edge NPU roadmaps across 5 vendors",
    "Summarize this private patient note"
]:
    print(q, "->", route_query(q, privacy_strict=("private" in q.lower())))